# Notebook 02 – Xây dựng đặc trưng

Gộp các bảng lịch sử về **một dòng cho mỗi hồ sơ**, tạo đặc trưng hành vi tín dụng, lọc cột và lưu `features_v1.parquet`.

> Bài toán gốc nhắc tới *lịch sử mua sắm và hành vi duyệt web*. Dữ liệu công khai không có hành vi duyệt web; lịch sử các khoản vay/trả góp trước đó (`applprev`, `credit_bureau`) được dùng làm đại diện cho hành vi.

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt, seaborn as sns
import utils
from utils import META

DATA_DIR, OUT_DIR = utils.get_data_dir(), utils.get_out_dir()
print("Du lieu:", DATA_DIR); print("Ket qua:", OUT_DIR)

## 1. Cấu hình

In [ ]:
SAMPLE_FRAC = 0.3       # may 8GB RAM: 30% ho so (~450k). None = toan bo (can may manh / Kaggle)
DEPTH0 = ["static_0", "static_cb_0"]
DEPTH1 = ["applprev_1", "person_1"]   # them "credit_bureau_a_1" neu du RAM (bang nang)

## 2. Gộp bảng
Bảng nhiều dòng/hồ sơ được gộp bằng: số bản ghi, max, mean (cột số) và giá trị của bản ghi đầu (cột phân loại).

In [ ]:
%%time
raw = utils.build_features(DEPTH0, DEPTH1, SAMPLE_FRAC)
print(raw.shape)

## 3. Đặc trưng nghiệp vụ bổ sung
Thêm tỷ lệ tự tạo ở đây sau khi đã xem tên cột thật (ví dụ nợ/thu nhập, khoản trả hàng tháng/thu nhập).

In [ ]:
def add_ratio(df, num, den, name):
    if num in df.columns and den in df.columns:
        return df.with_columns((pl.col(num) / pl.when(pl.col(den) > 0).then(pl.col(den))).alias(name))
    print("Bo qua", name, "- thieu cot"); return df

# Ten cot duoi day can doi chieu voi du lieu that (xem feature_definitions.csv cua cuoc thi)
raw = add_ratio(raw, "annuity_780A", "maininc_215A", "ratio_annuity_income")
raw = add_ratio(raw, "credamount_770A", "maininc_215A", "ratio_credit_income")
raw = add_ratio(raw, "annuity_780A", "credamount_770A", "ratio_annuity_credit")

## 4. Lọc cột và ép kiểu

In [ ]:
df = utils.prune_features(raw, max_null=0.95, max_card=200)
feats = [c for c in df.columns if c not in META]
print("Truoc:", raw.width, "cot | Sau:", df.width, "cot | So dac trung:", len(feats))
print("Bo nho: %.1f MB" % (df.estimated_size() / 1e6))

## 4b. Loại cột ngày phát sinh **sau** ngày quyết định (chống rò rỉ)
Cột ngày đã được đổi thành *số ngày so với ngày quyết định*; giá trị dương nghĩa là sự kiện xảy ra sau khi duyệt hồ sơ.
Quy tắc: cột có > 1% giá trị dương bị loại, **trừ** cột mà mô tả trong `feature_definitions.csv` cho biết đó là ngày đến hạn / ngày kết thúc dự kiến (biết trước tại thời điểm duyệt).
Bảng dưới liệt kê toàn bộ cột ngày có giá trị dương cùng quyết định giữ/loại – dùng trực tiếp cho phần phương pháp của báo cáo.

In [ ]:
defs = utils.load_definitions()
fut = utils.future_date_columns(df, feats, defs, min_share=0.01)
fut.to_csv(OUT_DIR / "future_date_check.csv", index=False)
DROP_FUTURE = fut.loc[fut["loai"], "cot"].tolist()
print("So cot ngay co gia tri sau ngay quyet dinh:", len(fut), "| loai:", len(DROP_FUTURE))
pd.set_option("display.max_colwidth", 90)
display(fut.round(4))
df = df.drop(DROP_FUTURE)
feats = [c for c in df.columns if c not in META]
print("So dac trung con lai:", len(feats))

## 4c. Loại biến bị **trôi về độ sẵn có** và thuộc tính nhạy cảm
**Trôi độ sẵn có.** Kiểm định adversarial ở phiên bản trước cho AUC = 1,0: một số cột (ví dụ `description_5085714M`, `contractssum_5085716L`, `requesttype_4525192L`, `birthdate_574D`) gần như **chỉ có dữ liệu ở giai đoạn sau** – ban tổ chức bắt đầu thu thập muộn. Mô hình học trên train (phần lớn rỗng) nhưng gặp giá trị đầy đủ ở giai đoạn sau → hành vi khó lường khi triển khai.
Quy tắc: so sánh tỷ lệ thiếu dữ liệu giữa **train (tuần đầu 70%) và valid (15% kế tiếp)** – không dùng tập test; loại cột có chênh lệch tỷ lệ thiếu ≥ 30 điểm phần trăm.

**Thuộc tính nhạy cảm.** Phân tích độ nhạy (notebook 06, phiên bản trước) cho thấy bỏ giới tính chỉ làm AUC test giảm 0,0019 (p = 0,012) nhưng thu hẹp chênh lệch tỷ lệ duyệt giữa nữ và nam từ 6,3 xuống 1,5 điểm phần trăm. Mô hình cuối vì vậy **không dùng giới tính** (đặt `EXCLUDE_SENSITIVE = False` để tái lập phiên bản cũ).

In [ ]:
EXCLUDE_SENSITIVE, MAX_NULL_SHIFT = True, 0.30
weeks = np.sort(df["WEEK_NUM"].unique()); n_w = len(weeks)
w_tr, w_va = weeks[: int(n_w * 0.70)], weeks[int(n_w * 0.70): int(n_w * 0.85)]
null_tr = df.filter(pl.col("WEEK_NUM").is_in(w_tr.tolist())).select([pl.col(c).is_null().mean().alias(c) for c in feats]).row(0)
null_va = df.filter(pl.col("WEEK_NUM").is_in(w_va.tolist())).select([pl.col(c).is_null().mean().alias(c) for c in feats]).row(0)
shift = pd.DataFrame({"thieu_train": null_tr, "thieu_valid": null_va}, index=feats)
shift["chenh"] = shift["thieu_valid"] - shift["thieu_train"]
shift["mo_ta"] = utils.describe(feats, defs)
DROP_SHIFT = shift.index[shift["chenh"].abs() >= MAX_NULL_SHIFT].tolist()
shift.sort_values("chenh", key=abs, ascending=False).to_csv(OUT_DIR / "availability_shift.csv")
print("Loai do troi do san co (|chenh ty le thieu| >= %.0f%%): %d cot" % (100 * MAX_NULL_SHIFT, len(DROP_SHIFT)))
display(shift.loc[DROP_SHIFT].sort_values("chenh", key=abs, ascending=False).round(3))

import re
SENS = re.compile(r"\bgender\b|\bsex\b", re.I)
DROP_SENS = [c for c in feats if EXCLUDE_SENSITIVE and SENS.search(utils.describe([c], defs).iloc[0] or "")]
print("Loai thuoc tinh nhay cam:", DROP_SENS)
df = df.drop(DROP_SHIFT + DROP_SENS)
feats = [c for c in df.columns if c not in META]
print("So dac trung con lai:", len(feats))

## 5. Xếp hạng đặc trưng theo IV (chỉ fit trên tập train)

In [ ]:
pdf = df.to_pandas()
train, valid, test = utils.time_split(pdf)
print("train/valid/test:", len(train), len(valid), len(test))
tr = train.sample(min(len(train), 300_000), random_state=42)
woe = utils.WoEEncoder().fit(tr[feats], tr["target"])
iv = woe.iv_.rename("IV").to_frame()
iv.to_csv(OUT_DIR / "iv_table.csv")
iv.head(30)

In [ ]:
leak = iv[iv["IV"] > 0.5]
print("Bien co IV > 0.5 (kiem tra ro ri truoc khi giu lai):"); display(leak)
DROP = []          # dien ten cot can loai sau khi kiem tra
df = df.drop(DROP)

## 6. Lưu kết quả

In [ ]:
df.write_parquet(OUT_DIR / "features_v1.parquet")
print("Da luu", OUT_DIR / "features_v1.parquet", df.shape)